# Reddit Scraper — Before Period
This notebook fetches the actual comment text from Reddit for the Brandwatch export.

**Input:** `before_reddit.csv` (Brandwatch export — no text)  
**Output:** `before_reddit_scraped.csv` (with actual comment text)

Run each cell one by one from top to bottom.

## Step 1 — Install and import packages

In [1]:
# Run this only once if requests or tqdm are not installed
# !pip install requests tqdm pandas

In [2]:
import csv
import io
import re
import time

import requests
import pandas as pd
from tqdm.notebook import tqdm

print('All packages imported successfully.')

All packages imported successfully.


## Step 2 — Load the Brandwatch CSV

In [3]:
# Load the Brandwatch export
# Brandwatch adds 5 metadata rows at the top before the real header
# so we need to find the real header row first

with open('before_reddit.csv', encoding='utf-8-sig') as f:
    raw = f.read()

reader = csv.reader(io.StringIO(raw))
rows = list(reader)

# Find the row that contains 'Url' and 'Sentiment' — that is the real header
header_idx = None
for i, row in enumerate(rows):
    if 'Url' in row and 'Sentiment' in row:
        header_idx = i
        break

headers = rows[header_idx]
data_rows = rows[header_idx + 1:]

df = pd.DataFrame(data_rows, columns=headers)
df = df[df['Url'].str.strip().astype(bool)].reset_index(drop=True)

print('Loaded', len(df), 'mentions')
df[['Url', 'Sentiment', 'Emotion']].head(3)

Loaded 350 mentions


,Url,Sentiment,Emotion
0,https://www.reddit.com/r/Coachella/comments/1k...,neutral,
1,https://www.reddit.com/r/popheads/comments/1sh...,neutral,Sadness
2,https://www.reddit.com/r/JUSTINBIEBER/comments...,negative,Anger


## Step 3 — Extract the URLs

In [4]:
# Get all Reddit URLs from the dataframe
urls = df['Url'].tolist()

print('Total URLs to scrape:', len(urls))
print('Example URL:', urls[0])

Total URLs to scrape: 350
Example URL: https://www.reddit.com/r/Coachella/comments/1k5dbwp/comment/ofgzsut/


## Step 4 — Define the scraping function

Reddit's public JSON API works by adding `.json` to any Reddit URL. No login or API key needed.

In [17]:
def fetch_comment(url, session):
    
    match = re.search(r'/comments/([A-Za-z0-9]+)/[^/]*/([A-Za-z0-9]+)', url)
    
    if not match:
        return {'url': url, 'comment_text': '', 'post_title': '',
                'subreddit': '', 'comment_score': '', 'created_utc': '', 'error': 'could not parse URL'}
    
    post_id = match.group(1)
    comment_id = match.group(2)
    
    json_url = 'https://www.reddit.com/comments/' + post_id + '.json?comment=' + comment_id
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Accept': 'application/json',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://www.reddit.com/',
    }
    
    try:
        response = session.get(json_url, headers=headers, timeout=15)
        
        if response.status_code == 429:
            print('Rate limited — waiting 65 seconds...')
            time.sleep(65)
            response = session.get(json_url, headers=headers, timeout=15)
        
        if response.status_code != 200:
            return {'url': url, 'comment_text': '', 'post_title': '',
                    'subreddit': '', 'comment_score': '', 'created_utc': '',
                    'error': 'HTTP ' + str(response.status_code)}
        
        data = response.json()
        
        post = data[0]['data']['children'][0]['data']
        post_title = post.get('title', '')
        subreddit = post.get('subreddit', '')
        post_score = post.get('score', '')
        created_utc = post.get('created_utc', '')
        
        comment_text = ''
        comment_score = ''
        
        def find_comment(listing, target_id):
            for child in listing.get('data', {}).get('children', []):
                d = child.get('data', {})
                if d.get('id') == target_id:
                    return d
                replies = d.get('replies', '')
                if isinstance(replies, dict):
                    result = find_comment(replies, target_id)
                    if result:
                        return result
            return None
        
        if len(data) > 1:
            found = find_comment(data[1], comment_id)
            if found:
                comment_text = found.get('body', '')
                comment_score = found.get('score', '')
                created_utc = found.get('created_utc', created_utc)
        
        return {
            'url': url,
            'subreddit': subreddit,
            'post_title': post_title,
            'comment_text': comment_text,
            'comment_score': comment_score,
            'post_score': post_score,
            'created_utc': created_utc,
            'error': ''
        }
    
    except Exception as e:
        return {'url': url, 'comment_text': '', 'post_title': '',
                'subreddit': '', 'comment_score': '', 'created_utc': '', 'error': str(e)}

print('Function defined successfully.')

Function defined successfully.


## Step 5 — Test on 3 URLs before running everything

Always test before running the full loop.

In [18]:
# Test on first 3 URLs only
session = requests.Session()

for url in urls[:3]:
    result = fetch_comment(url, session)
    print('URL:', url[:60])
    print('Subreddit:', result['subreddit'])
    print('Post title:', result['post_title'][:80])
    print('Comment:', result['comment_text'][:150])
    print('Error:', result['error'])
    print()
    time.sleep(1.5)

URL: https://www.reddit.com/r/Coachella/comments/1k5dbwp/comment/
Subreddit: 
Post title: 
Comment: 
Error: HTTP 403

URL: https://www.reddit.com/r/popheads/comments/1shwenw/comment/o
Subreddit: 
Post title: 
Comment: 
Error: HTTP 403

URL: https://www.reddit.com/r/JUSTINBIEBER/comments/1sfgun7/comme
Subreddit: 
Post title: 
Comment: 
Error: HTTP 403



In [9]:
import requests

r = requests.get(
    'https://www.reddit.com/r/JUSTINBIEBER.json',
    headers={'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}
)
print('Status:', r.status_code)
print('Response:', r.text[:300])

Status: 403
Response: <body class=theme-beta><div><style>.theme-light,:root{--rem360:22.5rem;--rem320:20rem;--rem192:12rem;--rem144:9rem;--rem128:8rem;--rem96:6rem;--rem90:5.625rem;--rem88:5.5rem;--rem64:4rem;--rem56:3.5rem;--rem48:3rem;--rem40:2.5rem;--rem36:2.25rem;--rem32:2rem;--rem28:1.75rem;--rem26:1.625rem;--rem24:


## Step 6 — Scrape all 350 URLs

This will take about 7–8 minutes (or longer if Reddit rate limits). The progress bar shows how far along it is.

**Do not close Jupyter while this is running.**

In [ ]:
results = []
session = requests.Session()

for url in tqdm(urls, desc='Fetching comments'):
    result = fetch_comment(url, session)
    results.append(result)
    time.sleep(1.5)

print('Done! Scraped', len(results), 'comments.')

## Step 7 — Merge with Brandwatch metadata and save

In [ ]:
# Convert results to a dataframe
scraped_df = pd.DataFrame(results)

# Keep useful columns from the original Brandwatch export
brandwatch_cols = ['Url', 'Sentiment', 'Emotion', 'Added']
brandwatch_cols = [c for c in brandwatch_cols if c in df.columns]
meta_df = df[brandwatch_cols].copy()

# Rename url column to match for merging
scraped_df = scraped_df.rename(columns={'url': 'Url'})

# Merge scraped text with Brandwatch metadata
final_df = scraped_df.merge(meta_df, on='Url', how='left')

# Extract subreddit from URL as a clean column (Brandwatch's column was empty)
final_df['subreddit_from_url'] = final_df['Url'].str.extract(r'reddit\.com/r/([^/]+)').iloc[:, 0].str.lower()

# Convert Unix timestamp to readable datetime
final_df['created_datetime'] = pd.to_datetime(
    pd.to_numeric(final_df['created_utc'], errors='coerce'), unit='s', utc=True
).dt.strftime('%Y-%m-%d %H:%M:%S UTC')

# Save to CSV
final_df.to_csv('before_reddit_scraped.csv', index=False, encoding='utf-8-sig')

print('Saved to before_reddit_scraped.csv')
print()

# Print a quick summary
total = len(final_df)
has_text = final_df['comment_text'].str.strip().astype(bool).sum()
deleted = total - has_text

print('Total rows:              ', total)
print('Comments with text:      ', has_text, f'({has_text/total*100:.1f}%)')
print('Deleted/empty comments:  ', deleted, f'({deleted/total*100:.1f}%)')
print()
print('Subreddit breakdown:')
print(final_df['subreddit_from_url'].value_counts().to_string())

## Step 8 — Preview the output

In [ ]:
# Show the first few rows of the final dataset
final_df[['subreddit_from_url', 'comment_text', 'Sentiment', 'Emotion', 'created_datetime']].head(10)

---
**Done.** The file `before_reddit_scraped.csv` is ready.

Load it in the analysis notebook with:
```python
df = pd.read_csv('before_reddit_scraped.csv')
```
The main text column for all analysis is `comment_text`.